# GalaxyGAN (cWGAN-GP) on Google Colab

**Runtime:** `Runtime` → `Change runtime type` → **GPU**.

**Colab Pro — which GPU?** For this project (69×69 cWGAN-GP), VRAM is not the bottleneck.

- **L4** (if offered): best **speed vs. compute units** for medium jobs — pick this for long training runs when available.
- **A100**: fastest iterations if you have compute units to spend; overkill for memory but nice for **shorter wall-clock** and larger batches.
- **T4**: still fine for this model; use if L4/A100 are unavailable.

See [Colab pricing / runtimes](https://cloud.google.com/colab/pricing) for how your plan bills compute units.

**First run:** downloads ~2.5 GB Galaxy10 HDF5 (Zenodo). Point outputs at **Google Drive** (cell below) so checkpoints survive disconnects.

In [ ]:
# Clone repo (public). For a private fork, use a GitHub token in the URL.
!rm -rf /content/GalaxyGAN
!git clone https://github.com/ndhanrajani/GalaxyGAN.git /content/GalaxyGAN
%cd /content/GalaxyGAN

In [ ]:
# PyTorch (Colab usually has torch; this ensures torchvision + deps for our code)
!pip install -q h5py scikit-learn torchvision

import torch
print("cuda?", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

In [ ]:
# Dataset (~2.54 GB) — only needed once per fresh runtime unless you persist data on Drive
import os
import subprocess
from pathlib import Path

data_dir = Path("/content/GalaxyGAN/data")
data_dir.mkdir(parents=True, exist_ok=True)
h5 = data_dir / "Galaxy10_DECals.h5"
url = "https://zenodo.org/records/10845026/files/Galaxy10_DECals.h5"

if not h5.is_file():
    subprocess.run(
        ["wget", "-O", str(h5), url],
        check=True,
    )
else:
    print("Already present:", h5)

os.environ["GALAXY10_H5"] = str(h5)
print("GALAXY10_H5=", h5)

In [ ]:
# Optional: mount Drive and write runs + checkpoints there
# from google.colab import drive
# drive.mount("/content/drive")
# OUT = "/content/drive/MyDrive/GalaxyGAN_runs/run1"
# os.makedirs(OUT, exist_ok=True)

OUT = "/content/GalaxyGAN/runs/colab_run"  # default: lost when runtime ends
import os
os.makedirs(OUT, exist_ok=True)
print("OUT=", OUT)

In [ ]:
import os
import subprocess
import sys

os.chdir("/content/GalaxyGAN")
# Smoke test: few epochs. Bump --epochs for real training (Colab may disconnect on long runs).
subprocess.run(
    [
        sys.executable,
        "-m",
        "galaxygan.train",
        "--epochs",
        "3",
        "--batch-size",
        "64",
        "--out",
        OUT,
    ],
    check=True,
)